# noise-batch-from-latent — worked example 3: Truncated normal latent batch via clamp

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `noise-batch-from-latent`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

Standard `t.randn` can occasionally produce large outlier values (e.g. ±4 or beyond), which can destabilize GAN generators. A practical trick is *truncated normal* sampling: draw from `N(0,1)` then clamp values to `[-threshold, +threshold]`. This keeps the latent vectors in a controlled range while retaining the bell-shaped distribution in the interior.

## Worked solution

**Step 1 — Draw standard-normal noise of shape `(B, L)`.**
We use `t.randn(batch_size, latent_dim, generator=g)` with a seeded generator for reproducibility.

**Step 2 — Clamp to `[-threshold, threshold]`.**
`t.clamp(noise, -threshold, threshold)` clips any element below `-threshold` up to `-threshold` and any element above `threshold` down to `threshold`. Elements already inside the range are untouched.

**Step 3 — Verify the clamp worked.**
We check that `noise.abs().max()` is ≤ `threshold`. We also verify the shape did not change — clamping is element-wise and shape-preserving.

**Step 4 — Observe the effect.**
With a tight threshold (e.g. 2.0), a fraction of draws get clipped. We print how many elements were clipped to make the effect visible.

In [ ]:
import torch as t

def truncated_noise(batch_size: int, latent_dim: int, threshold: float,
                   seed: int) -> t.Tensor:
    """Return (B, L) noise truncated to [-threshold, threshold]."""
    g = t.Generator()
    g.manual_seed(seed)
    noise = t.randn(batch_size, latent_dim, generator=g)
    return t.clamp(noise, -threshold, threshold)

# --- exercise it ---
B, L = 16, 128
threshold = 2.0

t.manual_seed(0)
g = t.Generator()
g.manual_seed(0)
noise_raw = t.randn(B, L, generator=g)

g2 = t.Generator()
g2.manual_seed(0)
noise_trunc = truncated_noise(B, L, threshold, seed=0)

clipped = (noise_raw.abs() > threshold).sum().item()
print(f'shape         : {noise_trunc.shape}')          # [16, 128]
print(f'max |value|   : {noise_trunc.abs().max().item():.4f}')  # ≤ 2.0
print(f'elements clipped: {clipped}')
assert noise_trunc.shape == (B, L)
assert noise_trunc.abs().max().item() <= threshold + 1e-6